In [1]:
import os
import pandas as pd
from datetime import datetime
import json
import gc

folder_path_demanddetails = '/home/prerna/Punjab/punjab-data-prod-analysis/chamkaursahib/output_demand_details/'

# read active properties & needed columns
property_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/chamkaursahib/eg_pt_property.csv',
    usecols=['id', 'propertyid', 'tenantid', 'createdtime', 'additionaldetails', 'ownershipcategory', 'status', 'usagecategory']
)
property_df = property_df[property_df['status'] == 'ACTIVE'].copy()

# read units
# unit_df = pd.read_csv(
#     '/home/prerna/Punjab/punjab-data-prod-analysis/srihargobindpur/eg_pt_unit.csv',
#     usecols=['propertyid', 'occupancytype']
# )



# read demand
demand_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/chamkaursahib/egbs_demand_v1.csv',
    dtype={"consumercode": str},
    low_memory=False,
    usecols=['id', 'taxperiodfrom', 'taxperiodto', 'consumercode', 'status', 'businessservice']
)
demand_df = demand_df[demand_df['status'] == 'ACTIVE'].copy()
demand_df = demand_df[demand_df['businessservice'] == 'PT'].copy()


# read demand details (memory‑efficient, in chunks)
# all_chunks = []
# needed_cols = ['demandid', 'taxamount', 'collectionamount', 'taxheadcode']
# for filename in os.listdir(folder_path_demanddetails):
#     if filename.endswith('.csv'):
#         file_path = os.path.join(folder_path_demanddetails, filename)
#         print(f'Loading: {file_path}')
#         chunk = pd.read_csv(file_path, usecols=needed_cols)
#         all_chunks.append(chunk)
demand_details_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/chamkaursahib/outputDemandDetails.csv',
    low_memory=False,
    usecols=['demandid', 'taxamount', 'collectionamount', 'taxheadcode']
)
# del all_chunks; gc.collect()

print("✅ Loaded data")

✅ Loaded data


In [2]:
print(len(property_df))         # number of rows in properties
# print(len(unit_df))             # number of rows in units
print(len(demand_df))   # number of rows in demand details
print(len(demand_details_df))   # number of rows in demand details

3356
14613
329136


In [3]:
# join demand and demand details
joined_demand = demand_df.merge(demand_details_df, left_on='id', right_on='demandid', how='left', suffixes=('_demand', '_detail'))
print(joined_demand['id'].nunique())
del demand_details_df, demand_df; gc.collect()
joined_demand.head()

14613


,id,consumercode,businessservice,taxperiodfrom,taxperiodto,status,demandid,taxheadcode,taxamount,collectionamount
0,25318,PT-2002-014095,PT,1522540800000,1554076799000,ACTIVE,25318,PT_OWNER_EXEMPTION,0.0,0.0
1,25318,PT-2002-014095,PT,1522540800000,1554076799000,ACTIVE,25318,PT_TIME_PENALTY,0.0,0.0
2,25318,PT-2002-014095,PT,1522540800000,1554076799000,ACTIVE,25318,PT_CANCER_CESS,18.0,18.0
3,25318,PT-2002-014095,PT,1522540800000,1554076799000,ACTIVE,25318,PT_FIRE_CESS,45.0,45.0
4,25318,PT-2002-014095,PT,1522540800000,1554076799000,ACTIVE,25318,PT_TAX,900.0,900.0


In [4]:
import pytz

# Correct: parse as datetime from milliseconds since epoch
joined_demand['taxperiodfrom'] = pd.to_datetime(joined_demand['taxperiodfrom'], unit='ms', utc=True)
joined_demand['taxperiodto'] = pd.to_datetime(joined_demand['taxperiodto'], unit='ms', utc=True)

# Convert to IST (Asia/Kolkata)
ist = pytz.timezone('Asia/Kolkata')
joined_demand['taxperiodfrom'] = joined_demand['taxperiodfrom'].dt.tz_convert(ist)
joined_demand['taxperiodto'] = joined_demand['taxperiodto'].dt.tz_convert(ist)

# Financial year calculation
def get_fy(date):
    if date.month >= 4:
        fy_start = date.year
        fy_end = date.year + 1
    else:
        fy_start = date.year - 1
        fy_end = date.year
    return f"{fy_start}-{str(fy_end)[-2:]}"

joined_demand['fy'] = joined_demand['taxperiodfrom'].apply(get_fy)

# Group by consumercode
result = joined_demand.groupby('consumercode')['fy'].agg(['min', 'max']).reset_index()
result.rename(columns={'min': 'earliest_fy', 'max': 'latest_fy'}, inplace=True)

print(result)

        consumercode earliest_fy latest_fy
0     PT-2002-011056     2018-19   2023-24
1     PT-2002-014069     2018-19   2023-24
2     PT-2002-014095     2018-19   2024-25
3     PT-2002-020623     2014-15   2023-24
4     PT-2002-020630     2014-15   2023-24
...              ...         ...       ...
3398  PT-2002-988755     2014-15   2023-24
3399  PT-2002-992047     2014-15   2023-24
3400  PT-2002-995680     2019-20   2023-24
3401  PT-2002-998548     2014-15   2023-24
3402  PT-2002-998574     2014-15   2023-24

[3403 rows x 3 columns]


In [5]:
# Merge latest_fy onto joined_demand by consumercode
joined = joined_demand.merge(
    result[['consumercode', 'latest_fy']],
    on='consumercode',
    how='left'
)

# Filter only latest FY
latest_demand = joined[joined['fy'] == joined['latest_fy']]

# Pivot taxheadcode values into separate columns
pivoted = latest_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
# PT_TAX + PT_CANCER_CESS + PT_FIRE_CESS + PT_ROUNDOFF - (PT_OWNER_EXEMPTION + PT_UNIT_USAGE_EXEMPTION)
pivoted['latest_fy_taxamount'] = (
    pivoted.get('PT_TAX', 0) +
    pivoted.get('PT_CANCER_CESS', 0) +
    pivoted.get('PT_FIRE_CESS', 0) +
    pivoted.get('PT_ROUNDOFF', 0) -
    ( pivoted.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Merge back into result
result = result.merge(
    pivoted[['consumercode', 'latest_fy_taxamount']],
    on='consumercode',
    how='left'
)

print(result.head())


     consumercode earliest_fy latest_fy  latest_fy_taxamount
0  PT-2002-011056     2018-19   2023-24               106.42
1  PT-2002-014069     2018-19   2023-24               170.79
2  PT-2002-014095     2018-19   2024-25               963.00
3  PT-2002-020623     2014-15   2023-24              2198.49
4  PT-2002-020630     2014-15   2023-24               978.46


In [6]:
# Calculating the tax amount (demand) of current year using formula
target_fy = "2025-26"
current_fy_demand = joined_demand[joined_demand['fy'] == target_fy]

# Pivot taxheadcode values into separate columns
pivoted_current = current_fy_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
pivoted_current['current_fy_taxamount'] = (
    pivoted_current.get('PT_TAX', 0) +
    pivoted_current.get('PT_CANCER_CESS', 0) +
    pivoted_current.get('PT_FIRE_CESS', 0) +
    pivoted_current.get('PT_ROUNDOFF', 0) -
    ( pivoted_current.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted_current.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Keep only required cols
pivoted_current = pivoted_current[['consumercode', 'current_fy_taxamount']]

# Ensure all consumercodes are present
all_consumercodes = pd.DataFrame(joined_demand['consumercode'].unique(), columns=['consumercode'])
final = all_consumercodes.merge(pivoted_current, on='consumercode', how='left')
final['current_fy_taxamount'] = final['current_fy_taxamount'].fillna(0)

# Merge into result
result = result.merge(final, on='consumercode', how='left')
result['current_fy_taxamount'] = result['current_fy_taxamount'].fillna(0)

print(result.head())


     consumercode earliest_fy latest_fy  latest_fy_taxamount  \
0  PT-2002-011056     2018-19   2023-24               106.42   
1  PT-2002-014069     2018-19   2023-24               170.79   
2  PT-2002-014095     2018-19   2024-25               963.00   
3  PT-2002-020623     2014-15   2023-24              2198.49   
4  PT-2002-020630     2014-15   2023-24               978.46   

   current_fy_taxamount  
0                   0.0  
1                   0.0  
2                   0.0  
3                   0.0  
4                   0.0  


In [7]:
property_result_merged = property_df.merge(
    result,
    left_on='propertyid',
    right_on='consumercode',
    how='left'
)

print(property_result_merged)

                                        id       propertyid          tenantid  \
0     24affb1d-5740-49bd-8ebc-844ee573ea2b  PT-2002-2008857  pb.chamkaursahib   
1     cf46cb05-c777-4ee8-96aa-b75b429fb715  PT-2002-2032655  pb.chamkaursahib   
2     e8626877-333d-4b6a-bee1-d975ca6a207c  PT-2002-2109515  pb.chamkaursahib   
3     f0c3431a-5fdb-43aa-8940-004584f8673b  PT-2002-1031373  pb.chamkaursahib   
4     19fd36fb-2efd-4407-ab67-a706906259cb  PT-2002-1033429  pb.chamkaursahib   
...                                    ...              ...               ...   
3351  1296c4d2-3458-4b07-a5dc-ec8286ebfb72  PT-2002-1625859  pb.chamkaursahib   
3352  7336502e-7e09-401c-b16e-bb6f569cc9af  PT-2002-1696248  pb.chamkaursahib   
3353  e6dd6cfd-b013-4d99-9f92-2c6c88e620f0  PT-2002-1625870  pb.chamkaursahib   
3354  c6f7fc05-a7c2-4736-8c27-1c4dc6fa2abc  PT-2002-1986982  pb.chamkaursahib   
3355  97c28894-e91c-4027-8660-41d2039c4658  PT-2002-1993793  pb.chamkaursahib   

      status          owner

In [8]:
property_result_merged.to_csv('Punjab_Data_Analysis_chamkaursahib_final_2.csv', index=False)